# Step 06. 판단 시나리오 2 — TTC 기반 Decision

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

**TTC(Time-To-Collision)** 지표로 위험을 정량화하고, 임계값 기반 Decision을 구성합니다.  
Day 1·2에서 만든 Planning(경로) 알고리즘과 연결되는 상위 판단 계층입니다.

> ⚠️ **이 노트북의 CARLA 코드 셀은 시뮬레이터가 실행 중이어야 동작합니다.**  
> `./CarlaUE4.sh` 로 서버를 켠 뒤 위에서부터 실행하세요.

## 학습 목표

1. TTC 정의와 계산
2. 임계값 기반 Decision (KEEP/BRAKE/EMERGENCY)
3. 합성 데이터로 로직 검증 (서버 없이)
4. CARLA 라이브에서 ego-lead TTC 계산·대응
5. Planning(Day1/2) 과의 연결

## 1. TTC와 Decision 로직 (합성 데이터)

### 이론 배경

$$TTC = \frac{\text{전방 간격}}{\text{접근 속도}} = \frac{gap}{v_{ego} - v_{lead}}$$

접근 속도가 양수(가까워짐)일 때만 유효하며, 작을수록 위험합니다.  
임계값으로 행동을 결정합니다: `TTC<1.5s` → 비상제동, `<3s` → 제동, `<5s` → 차선변경 검토.  

아래는 **서버 없이 실행 가능한** 합성 시나리오로 로직을 먼저 검증합니다.

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt

# TTC(Time-To-Collision) = 전방 간격 / 접근 속도 (closing speed)
def compute_ttc(gap, rel_speed):      # rel_speed = v_ego - v_lead (>0 이면 접근)
    if rel_speed <= 1e-3: return float('inf')   # 멀어지거나 동속이면 충돌 위험 없음
    return gap / rel_speed

def decide(ttc, gap):                 # 간단한 임계값 기반 Decision
    if ttc < 1.5 or gap < 5:  return 'EMERGENCY_BRAKE'
    if ttc < 3.0:             return 'BRAKE'
    if ttc < 5.0:             return 'LANE_CHANGE?'
    return 'KEEP'

# 합성 시나리오: ego 60km/h, lead가 t=2s에 급감속 -> 간격 급감
t = np.arange(0, 8, 0.1)
v_ego = np.full_like(t, 60/3.6)
v_lead = np.where(t < 2, 50/3.6, np.maximum(50/3.6 - 6*(t-2), 0))   # 6 m/s^2 감속
gap = 30 + np.cumsum((v_lead - v_ego)) * 0.1
gap = np.maximum(gap, 0.1)
ttc = [compute_ttc(g, ve-vl) for g, ve, vl in zip(gap, v_ego, v_lead)]
dec = [decide(tt, g) for tt, g in zip(ttc, gap)]

fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t, gap, '-b'); ax[0].set_ylabel('gap [m]'); ax[0].grid(alpha=0.3); ax[0].set_title('Synthetic cut-in / hard-brake scenario')
ax[1].plot(t, [min(x,10) for x in ttc], '-r'); ax[1].axhline(1.5, ls='--', c='k', alpha=0.5); ax[1].axhline(3.0, ls=':', c='gray')
ax[1].set_ylabel('TTC [s] (capped 10)'); ax[1].grid(alpha=0.3)
colors={'KEEP':'green','LANE_CHANGE?':'gold','BRAKE':'orange','EMERGENCY_BRAKE':'red'}
ax[2].scatter(t, [0]*len(t), c=[colors[d] for d in dec], s=30); ax[2].set_yticks([])
ax[2].set_xlabel('time [s]'); ax[2].set_title('Decision (green KEEP / orange BRAKE / red EMERGENCY)')
plt.tight_layout(); plt.show()
print('TTC가 임계값(3s, 1.5s) 아래로 떨어지면 BRAKE/EMERGENCY로 전환되는 로직')


## 2. CARLA 라이브 TTC 계산

### 이론 배경

Step 05에서 spawn한 `ego`, `lead` 의 상태를 매 틱 읽어 TTC를 계산합니다.  
- 속도: `get_velocity()` 의 크기
- 간격: `get_location().distance()` 에서 차량 길이 보정

Decision 결과를 `VehicleControl` 로 ego에 적용합니다 (제동).  
> 이 셀은 Step 05의 ego/lead가 살아있는 상태에서 이어서 실행하세요.

In [ ]:
# 라이브 TTC 계산 + Decision + ego 제어는 동기 tick 루프가 필요 -> 스크립트로 실행
print('터미널에서 실행: python3 scripts/run_planning_carla.py')
print('  ego-lead 상황에서 매 틱 TTC 계산 -> BRAKE/EMERGENCY 판단 -> ego 제어')
print('  종료 후 ttc_log.png (gap/TTC 그래프) 가 저장됩니다')

## 3. 라이브 로그 시각화

### 이론 배경

기록한 gap·TTC를 시간축으로 그려, 합성 시나리오(섹션 1)와 같은 패턴이 나오는지 확인합니다.

In [ ]:
# run_planning_carla.py 가 저장한 output/ttc_log.png 를 확인
from IPython.display import Image, display
import os
path = 'scripts/output/ttc_log.png'
if os.path.exists(path):
    display(Image(path))
else:
    print('먼저 터미널에서 실행하세요: python3 scripts/run_planning_carla.py')
    print('  -> scripts/output/ttc_log.png, ttc_log.csv 생성')

## 4. Planning과의 연결

### 이론 배경

Decision(상위 판단)이 행동을 정하면, **실제 경로는 Day 1·2의 Planning이 생성**합니다.

```
Perception -> Prediction -> [Decision: TTC 판단] -> [Planning: 경로 생성] -> Control
                              (이 노트북)            (Day1 A*/Hybrid,
                                                     Day2 Frenet/RRT)
```

- `BRAKE` → 종방향 속도 프로파일 재계획 (Day2 Velocity Planning)
- `LANE_CHANGE` → Frenet d 오프셋 변경으로 후보 경로 생성 (Day2 Frenet)
- 좁은 공간 회피 → Hybrid A* / RRT (Day1·2)

즉, TTC 판단이 **어떤 Planner를 어떤 목표로 호출할지**를 결정하는 스위치 역할을 합니다.